# Introduction ✨



![](https://drive.google.com/uc?export=view&id=1TuesF83uT3BoShpMgIW5NN2itlNIwGX-)
![](https://drive.google.com/uc?export=view&id=11FoXiDS0XcQG5R9zUh6luvjfJxxQ3vYT)
![](https://drive.google.com/uc?export=view&id=145LaNxAZsxPzXoOuFOxJxak_1J90fs5l)

This notebook demonstrates how to train models for the [AIDA-X](https://github.com/AidaDSP/aida-x) plugin. If run inside of Colab, it will automatically use a free Google Cloud GPU.

At the end, you'll have a custom-trained model that you can download and play directly on AIDA-X plugin.\
[DEMO VIDEO]() 🔊🔊🔊

---
This notebook is brought to you collaboration between the [MOD Audio](https://mod.audio) and the [AIDA DSP](https://aidadsp.github.io) teams.\
Some of the code and workflow presented here is inspired by the [NAM](https://github.com/sdatkinson/neural-amp-modeler) training [colab](https://colab.research.google.com/github/sdatkinson/neural-amp-modeler/blob/main/bin/train/easy_colab.ipynb?authuser=1#scrollTo=5CQleTk7GJV8) notebook.

---  


## **Instructions** ([step-by-step video](https://www.youtube.com/watch?v=htpK0QLzeKA))
Whenever you see `<- RUN CELL (►)`, you need to press the (►) next to it, to run the code that will fulfill that step.  

> The steps in this notebook are pretty straightforward:
0.   Deps 👾
1.   Set-up 👾
2.   Data 📑
3.   Model Training 🏋️‍♂️
4.   Model Evaluation 📈 (optional)
5.   Model Export ✅






# 0. Deps 👾

In [ ]:
# Check PyTorch and CUDA versions
import torch
import re

pytorch_version = torch.__version__
cuda_version = torch.version.cuda

required_pytorch_version = "2.3.1"
required_cuda_version = "12.1"

def version_higher(version1, version2):
  def extract_numeric_version(version):
    return tuple(map(int, re.findall(r'\d+', version)))
  return extract_numeric_version(version1) > extract_numeric_version(version2)

if version_higher(pytorch_version, required_pytorch_version) or version_higher(cuda_version, required_cuda_version):
  print(f"WARNING: Your environment has PyTorch {pytorch_version} and CUDA {cuda_version}. This environment is not supported.")
  print("Proceeding to install required dependencies...")
  !pip3 uninstall --disable-pip-version-check -y torch torchvision torchaudio
  !pip3 install --disable-pip-version-check --no-cache-dir \
    torch==2.3.1+cu121 \
    torchvision==0.18.1+cu121 \
    torchaudio==2.3.1+cu121 \
    -f https://download.pytorch.org/whl/torch_stable.html
  print("PyTorch and CUDA versions have been set to the required versions. Please restart the runtime.")

# 1. Set-up 👾

In [ ]:
#@markdown `<- RUN CELL (►)`

#@markdown This will check for GPU availability, prepare the code for you, and mount your drive.

import torch
import os
import numpy as np
import IPython
from time import sleep
import librosa

print("---")
if 'step' in locals():
  print("Ready! you can now move to step 1: DATA")
else:

  print("Checking GPU availability...", end=" ")
  if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU available! ")
  else:
    device = torch.device("cpu")
    print("GPU unavailable, using CPU instead.")
    print("RECOMMENDED: You can enable GPU through \"Runtime\" -> \"Change runtime type\" -> \"Hardware accelerator:\" GPU -> Save")

  if any(key.startswith("COLAB_") for key in os.environ):
    if not os.path.exists("/content/Automated-GuitarAmpModelling"):
      print("Getting the code...")
      !git clone https://github.com/pilali/Automated-GuitarAmpModelling.git &>> /content/log.txt
      assert os.path.exists("/content/Automated-GuitarAmpModelling"), f"Error getting the code!"

      os.chdir('/content/Automated-GuitarAmpModelling')
      !git checkout next &>> /content/log.txt

      print("Checking for code updates...")
      !git submodule update --init --recursive &>> /content/log.txt

      print("Installing dependencies...")
      !pip3 install --disable-pip-version-check --no-cache-dir auraloss==0.4.0 &>> /content/log.txt

      print("Mounting google drive...")
      from google.colab import drive
      drive.mount('/content/drive')
    else:
      print("Code already exists. Skipping Google Drive mounting.")
  else:
    print("Not running on Google Colab. Skipping Colab-specific setup.")

  # Adjust the env
  os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:2"

  from colab_functions import wav2tensor, extract_best_esr_model, create_csv_aidax
  from prep_wav import WavParse
  import plotly.graph_objects as go
  from CoreAudioML.networks import load_model
  import CoreAudioML.miscfuncs as miscfuncs
  if any(key.startswith("COLAB_") for key in os.environ):
    from google.colab import files
  import io
  import shutil
  os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

  step = 0
  print()
  print("Ready! you can now move to step 2: DATA")

# 2. The Data (upload + preprocessing) 📑

### Step 2.1: Download the capture signal
Download the pre-crafted "capture signal" called [input.wav](https://drive.google.com/file/d/1TNpaPPc9tdCu6OA1VETWvufc7wG2nQTJ/view?usp=sharing) from the provided link.

### Step 2.2 Reamp your gear
Use the downloaded capture signal to reamp the gear that you want to model. Record the output and save it as "target.wav".
For a detailed demonstration of how to reamp your gear using the capture signal, refer to this [video tutorial](https://youtu.be/lrvuODtk9W0?t=70) starting at 1:10 and ending at 3:44.

In [ ]:
#@markdown `<- RUN CELL (►)`

#@markdown Step 1.3 upload
#@markdown ---
#@markdown * In drive, put the 2 audio files with which you would like to train in a single folder.
#@markdown  * `input.wav` : contains the reference (dry/DI) sound.
#@markdown  * `target.wav` : contains the target (amped/with effects) sound.
#@markdown * Use the file browser in the left panel to find a folder with your audio, right-click **"Copy Path", paste below**, and run the cell.
#@markdown  * ex. `/content/Automated-GuitarAmpModelling/Recordings`
#@markdown For dynamic models requiring multiple audio files, those files should be specified within your custom JSON config (see advanced options below). The `input.wav` and `target.wav` copied from `DATA_DIR` will be used if your custom JSON refers to `./input.wav` and `./target.wav` as one of its dataset entries, or when using the default configuration for non-dynamic training.
DATA_DIR = '' #@param {type: "string"}
#@markdown ---
#@markdown ### Advanced: Custom Configuration for Data Processing
#@markdown Optionally, provide the path to your custom JSON configuration file for `prep_wav.py`.
#@markdown This is necessary if you want to prepare data for **dynamic models**, which require a specific JSON structure:
#@markdown - The JSON should contain a `params` object with:
#@markdown   - `"n"`: an integer for the number of dynamic parameters.
#@markdown   - `"datasets"`: a list of objects, where each object defines an `input` audio path, a `target` audio path, and a list of `params` values for that recording.
#@markdown - Ensure paths within your JSON are correct (e.g., relative to `/content/Automated-GuitarAmpModelling/` or absolute Colab paths like `/content/drive/MyDrive/...`).
#@markdown If left as `Configs/Default.json`, it will prepare for standard, non-dynamic model training using the `input.wav` and `target.wav` from `DATA_DIR`.
USER_CONFIG_PATH = 'Configs/Default.json' #@param {type: "string"}

assert 'step' in locals(), "Please run the code in the introduction section first!"
print("---")
assert DATA_DIR != '', "Please input a path for your DATA_DIR"
assert os.path.exists(DATA_DIR), f"Drive Folder Doesn\'t Exists: {DATA_DIR}"
assert set(["input.wav", "target.wav"]) <= set([x.lower() for x in os.listdir(DATA_DIR)]), \
  "Make sure you have \"input.wav\" and \"target.wav\" inside your data folder"

# Copy the files to /content/ and overwrite if they already exist using bash commands
destination_dir = "/content/Automated-GuitarAmpModelling"
input_path = os.path.join(destination_dir, "input.wav")
target_path = os.path.join(destination_dir, "target.wav")

!cp -f "{os.path.join(DATA_DIR, 'input.wav')}" "{input_path}"
print(f"File copied: {input_path}")

!cp -f "{os.path.join(DATA_DIR, 'target.wav')}" "{target_path}"
print(f"File copied: {target_path}")

# Create the CSV and parse the WAV files
create_csv_aidax("/content/Automated-GuitarAmpModelling/Configs/Csv/modaudioug.csv")
WavParse(load_config=USER_CONFIG_PATH, config_location='/content/Automated-GuitarAmpModelling', norm=False, denoise=False)

step = max(step, 1)
print()
print("Data prepared! You can now move to step 3: TRAINING")

# 3. Model Training 🏋️‍♂️

In [ ]:
#@markdown `<- RUN CELL (►)`

#@markdown Training usually takes around 10 minutes,
#@markdown but this can change depending on the duration of
#@markdown the training data that you provided and the model_type
#@markdown you choose.\
#@markdown Note that training doesn't always lead to the same results.
#@markdown You may want to run it a couple of times and compare the results.

#@markdown Choose the Model type you want to train:\
#@markdown Generally, the heavier the model the more accurate it is, but also the more CPU it consumes.
#@markdown Here's a list of approximate CPU consumption of each model type on a [MOD Dwarf](https://mod.audio/dwarf/):
#@markdown * Lightest: 25% CPU
#@markdown * Light: 30% CPU
#@markdown * Standard: 37% CPU
#@markdown * Heavy: 46% CPU
model_type = "Standard" #@param ["Lightest", "Light", "Standard", "Heavy"]
#@markdown Some training hyper parameters
#@markdown (Recommended: ignore and continue with default values):
skip_connection = "OFF" #@param ["ON", "OFF"]
epochs = 200 #@param {type:"slider", min:100, max:2000, step:20}
print("---")

if model_type == "Lightest":
  config_file = "LSTM-8-1"
elif model_type == "Light":
  config_file = "LSTM-12-1"
elif model_type == "Standard":
  config_file = "LSTM-16-1"
elif model_type == "Heavy":
  config_file = "LSTM-20-1"

if skip_connection == "ON":
  skip_con = 1
else:
  skip_con = 0

assert 'step' in locals(), "Please run the code in the introduction section first!"
assert step>=1, "Please execute the \"1.DATA\" cell code to prepare the data for the training!"

!python3 dist_model.py -l "$config_file" -lm 0 -sc $skip_con -eps $epochs

sleep(1)
model_dir = f"/content/Automated-GuitarAmpModelling/Results/MOD-AUDIO-UG"
step = max(step, 2)
print("Training done!\nESR after training: ", extract_best_esr_model(model_dir)[1])
print("You can now move to step 4: EVALUATION or directly to step 5: EXPORT")

# 4. Model Evaluation 📈


In [ ]:
#@markdown `<- RUN CELL (►)`

#@markdown Here you can visualize and listen to the output of your trained model on the data you provided earlier.

assert 'step' in locals(), "Please run the code in the introduction section first!"
assert step>=1, "Please execute the \"1.DATA\" cell code to prepare the data for the training!"
assert step>=2, "Please execute the \"2.TRAINING\" cell code to train a model for evaluation!"

print("---")
# Find the file with .full_name extension in model_dir
full_name_file = [f for f in os.listdir(model_dir) if f.endswith('.full_name')]
assert len(full_name_file) == 1, "There should be exactly one file with the .full_name extension in the model_dir."

# Remove the .full_name extension to create the model_filename
model_filename = os.path.splitext(full_name_file[0])[0] + '.aidax'

# Extract the best model available from training results
model_path, esr = extract_best_esr_model(model_dir)
model_data = miscfuncs.json_load(model_path)
model = load_model(model_data).to(device)

full_dry = wav2tensor(f"/content/Automated-GuitarAmpModelling/Data/test/aidadsp-auto-input.wav")
full_amped = wav2tensor(f"/content/Automated-GuitarAmpModelling/Data/test/aidadsp-auto-target.wav")

samples_viz = 24000
duration_audio = 5
seg_length = int(duration_audio * 48000)
start_sample = np.random.randint(len(full_dry)-duration_audio*48000)
dry = full_dry[start_sample:start_sample+seg_length]
amped = full_amped[start_sample:start_sample+seg_length]
with torch.no_grad():
  modeled = model(dry[:, None, None].to(device)).cpu().flatten().detach().numpy()

print(f"Current model: {model_filename}")
print(f"ESR:", esr)
# Visualization
fig = go.Figure()
fig.add_trace(
  go.Scatter(
    x=list(np.arange(len(dry[:samples_viz]))/48000), y=dry[:samples_viz],
    name="dry", mode='lines'
  )
)
fig.add_trace(
  go.Scatter(
    x=list(np.arange(len(amped[:samples_viz]))/48000), y=amped[:samples_viz],
    name="target", mode='lines'
  )
)
fig.add_trace(
  go.Scatter(
    x=list(np.arange(len(modeled[:samples_viz]))/48000), y=modeled[:samples_viz],
    name="prediction", mode='lines'
  )
)
fig.update_layout(
  title="Dry vs Target vs Predicted signal",
  xaxis_title="Time (s)",
  yaxis_title="Signal Amplitude",
  legend_title="Signal",
)
fig.show()

# Listen
print("DRY Signal:")
IPython.display.display(IPython.display.Audio(data=dry, rate=48000))

print("TARGET Signal:")
IPython.display.display(IPython.display.Audio(data=amped, rate=48000))

print("PREDICTED Signal:")
IPython.display.display(IPython.display.Audio(data=modeled, rate=48000))

print("Difference Signal:")
difference_signal = np.array(amped) - np.array(modeled)
IPython.display.display(IPython.display.Audio(data=difference_signal, rate=48000))

# Cleanup
del dry, amped, modeled, full_dry, full_amped, model
torch.cuda.empty_cache()

step = max(step, 3)

In [ ]:
#@markdown `<- RUN CELL (►)`

#@markdown Here you can **upload** your own dry guitar files, and listen to the predicted output of the model.
#@markdown (Optional) If using a dynamic model, enter comma-separated parameter values (e.g., "0.5,0.2")
DYNAMIC_PARAMS = '' #@param {type: "string"}

assert 'step' in locals(), "Please run the code in the introduction section first!"
assert step>=1, "Please execute the \"1.DATA\" cell code to prepare the data for the training!"
assert step>=2, "Please execute the \"2.TRAINING\" cell code to train a model for evaluation!"

print("---")
import os
import IPython.display
import io # Should already be imported from setup cell, but good to ensure
from colab_functions import extract_best_esr_model # Ensure this is available

# Determine model_path (re-calculate or ensure it's passed from previous cell)
# Assuming model_dir is defined in the training cell and accessible here.
# If not, model_dir might need to be re-established or passed.
if 'model_dir' not in locals():
    # This is a fallback if model_dir is not in the current scope.
    # It assumes a default naming convention if the training cell (h6RdceOeWdZl) was run.
    # A more robust solution would be to ensure model_dir is correctly passed or stored.
    print("Warning: 'model_dir' not found in local scope. Attempting to reconstruct. This might fail if training steps changed.")
    model_dir = f"/content/Automated-GuitarAmpModelling/Results/MOD-AUDIO-UG"
    if not os.path.exists(model_dir):
        raise FileNotFoundError(f"Model directory {model_dir} not found. Please ensure the training step (cell id h6RdceOeWdZl) was run successfully.")
model_path, _ = extract_best_esr_model(model_dir) # _ is esr, not needed here
if not model_path or not os.path.exists(model_path):
    raise FileNotFoundError(f"Best model JSON not found at '{model_path}'. Please ensure training completed and model_dir is correct.")

if any(key.startswith("COLAB_") for key in os.environ):
  # Ensure google.colab.files is imported if in Colab
  try:
    from google.colab import files
    uploaded = files.upload()
  except ImportError:
    print("Error: 'google.colab.files' not available. Are you running in a Colab environment?")
    uploaded = {} # Initialize to empty dict to avoid further errors
else:
  print("Not in Colab environment. Skipping file upload. You'll need to manually provide files if you intend to process them.")
  uploaded = {} # Initialize to empty dict

print()
print("Running predictions using proc_audio.py:")

for k, v_bytes in uploaded.items():  # v_bytes are the file contents as bytes
  print("##### Processing:", k)
  temp_input_path = f"/tmp/{k.replace(' ', '_')}" # Sanitize filename for temp storage
  temp_output_path = f"/tmp/output_{k.replace(' ', '_')}"

  try:
    with open(temp_input_path, 'wb') as f:
      f.write(v_bytes)

    cmd = f"python3 /content/Automated-GuitarAmpModelling/proc_audio.py -l {model_path} -i {temp_input_path} -o {temp_output_path}"
    if DYNAMIC_PARAMS and DYNAMIC_PARAMS.strip():
      cmd += f" -pv \"{DYNAMIC_PARAMS}\"" # Add params, ensure quotes for safety

    print(f"Executing: {cmd}")
    # Using os.system might be simpler for capturing output in Colab if !{cmd} has issues with variable expansion sometimes
    # However, !{cmd} is generally preferred for direct shell execution display in notebooks.
    exit_code = os.system(f"{cmd} > /tmp/proc_audio_output.txt 2>&1") # Redirect output
    with open("/tmp/proc_audio_output.txt", "r") as outfile:
        print(outfile.read()) # Print the output of the script
    if exit_code != 0:
        print(f"Error processing {k}. proc_audio.py exited with code {exit_code}. Check output above.")
        continue # Skip to next file if error

    print(f"\n--- Original Uploaded Audio: {k} ---")
    # To get the actual rate, it's better to load it with librosa or soundfile
    # For simplicity, assuming 48kHz as per original, but this might not always be true.
    # IPython.display.Audio(data=v_bytes, rate=48000) # rate should ideally be determined from file
    IPython.display.display(IPython.display.Audio(filename=temp_input_path))

    if os.path.exists(temp_output_path):
      print(f"--- Processed Audio: output_{k} ---")
      IPython.display.display(IPython.display.Audio(filename=temp_output_path))
    else:
      print(f"Output file {temp_output_path} not found. Processing might have failed.")

  except Exception as e:
    print(f"An error occurred while processing {k}: {e}")
  finally:
    # Cleanup temporary files
    if os.path.exists(temp_input_path):
      os.remove(temp_input_path)
    if os.path.exists(temp_output_path):
      os.remove(temp_output_path)
    if os.path.exists("/tmp/proc_audio_output.txt"):
      os.remove("/tmp/proc_audio_output.txt")

step = max(step, 3) # Assuming this step is part of a sequence
torch.cuda.empty_cache() # Clear CUDA cache if torch was used significantly before this cell by other means


# 5. Model Export ✅

In [ ]:
#@markdown `<- RUN CELL (►)`

#@markdown Download a .aidax file summarizing the model that you just trained.

#@markdown You can then upload it to AIDA-X model loader plugin and run it in real-time.

assert 'step' in locals(), "Please run the code in the introduction section first!"
assert step>=1, "Please execute the \"1.DATA\" cell code to prepare the data for the training!"
assert step>=2, "Please execute the \"2.TRAINING\" cell code to train a model for evaluation!"

print("---")
# Find the file with .full_name extension in model_dir
full_name_file = [f for f in os.listdir(model_dir) if f.endswith('.full_name')]
assert len(full_name_file) == 1, "There should be exactly one file with the .full_name extension in the model_dir."

# Remove the .full_name extension to create the model_filename
model_filename = os.path.splitext(full_name_file[0])[0] + '.aidax'

print("Generating model file:", model_filename)

# Extract the best model available from training results
model_path, esr = extract_best_esr_model(model_dir)
!python3 modelToRTNeural.py -l "$config_file" -ax

# Define the destination directory
destination_path = os.path.join(DATA_DIR, model_filename)

# Copy the generated file to the destination directory
!cp "{os.path.join(model_dir, 'model_rtneural.aidax')}" "{destination_path}"

if any(key.startswith("COLAB_") for key in os.environ):
  from google.colab import files
  files.download(destination_path)

print()
print("Model file saved to:", destination_path)
step = max(step, 4)